# 김건우 개인 실습 — 공개 정리본
교육 과정의 제공 골격을 바탕으로 수행한 실습입니다. 개인/팀 범위를 폴더로 구분했습니다.
공개 정리 시 환경 연결 셀·데이터 미리보기·내장 교안 이미지를 제외했습니다. 학습 설정과 모델 로직은 원본을 유지합니다.
개인 노트북의 평가 텍스트는 당시 저장 결과이며 나머지 실행 출력은 정리했습니다.


In [ ]:
import os


# **Step2. Multi-label 분류**



**미션 : 리뷰에 포함된 11가지 속성을 검출하기(Multi-label Multi-Classification)**

1) 목표 : 리뷰 내용에서, 속성에 해당하는 내용이 등장하는지 분류하는
모델 생성
2) 절차
    * 데이터 전처리 : 주어진 데이터를 목표에 맞는 형태로 변환
    * 모델링 : 다중 레이블 분류 파인 튜닝
        * 일반 파인 튜닝
        * LoRA 파인 튜닝

## 1.환경준비

### (1) 라이브러리 설치 및 로딩

In [ ]:
!pip install datasets peft accelerate -q

In [ ]:
# 한글폰트 설치
import matplotlib as mpl
import matplotlib.font_manager as fm

%config InlineBackend.figure_format = 'retina'
!apt -qq -y install fonts-nanum

font_path = '/usr/share/fonts/truetype/nanum/NanumBarunGothic.ttf'
fontprop = fm.FontProperties(fname=font_path)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import ast
import torch

from transformers import Trainer, TrainingArguments, EarlyStoppingCallback, AutoTokenizer, AutoModelForSequenceClassification
from datasets import load_dataset, Dataset

from sklearn.model_selection import train_test_split
from sklearn.metrics import *
from sklearn.preprocessing import LabelEncoder

from peft import get_peft_model, LoraConfig, TaskType

import logging
logging.getLogger('matplotlib.font_manager').setLevel(logging.ERROR)

### (2) GPU 설정
* GPU 사용을 위한 설정

In [ ]:
# GPU 설정
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

## 2.데이터 준비

### (1) 데이터 로딩
* data.csv 파일 로딩

In [ ]:
# 공개본: 환경 연결 셀은 생략. 데이터 경로는 DATA_PATH로 지정합니다.


In [ ]:
data = pd.read_csv(os.environ.get("DATA_PATH", "../../data/data.csv"))
data.head()

### (2) Dataset 준비1

In [ ]:
# 레이블 리스트
labels = ['보습', '기능', '밀착', '가격', '향', '편의성', '기타', '외형', '포장', '품질', '피부']

* multi-hot label 만들기
    * multi-hot label 값이 문자열이 아닌 리스트여야 합니다.
* multi-hot label을 추가한 데이터셋을 data1 으로 저장합니다.

In [ ]:
grouped_data = data.groupby('text')['aspect'].apply(lambda x: list(set(x)))
display(grouped_data.head())

In [ ]:
from sklearn.preprocessing import MultiLabelBinarizer

mlb = MultiLabelBinarizer(classes=labels)

encoded_labels = mlb.fit_transform(grouped_data)
list_of_labels = [list(row) for row in encoded_labels]

data1 = pd.DataFrame({
    'text': grouped_data.index,
    'multi_hot_label': list_of_labels
})

data1 = data1.reset_index(drop=True)

* 검증 : 제대로 생성되었는지 다음 코드로 검증해 봅니다.

In [ ]:
print(data1.columns)

In [ ]:
review = "(장점 ) 1.쓰기편하고 제형이 묽어서 발림성 좋다. 2.백탁없다 (단점)"
display(data[data['text'].str.startswith(review)])
display(data1[data1['text'].str.startswith(review)])

* 전체 데이터에서 10000건을 샘플링 합니다.(파인튜닝 시간 절약을 위해서)

In [ ]:
data_sample = data1[:3000]
data_sample.shape

### (3) Dataset 준비2
* train, val 분할 7:3
* 텐서 데이터셋

In [ ]:
# train, val 분할 7:3
train, val = train_test_split(data_sample, test_size=0.3, random_state=100)

In [ ]:
# df로 부터 텐서 데이터셋 만들기
train_ts = Dataset.from_pandas(train)
val_ts = Dataset.from_pandas(val)

In [ ]:
train_ts[:5]

### (4) 토크나이징

* 다중 레이블 분류 문제에서 label은 float 타입이어야 함

In [ ]:
# 토크나이저 다운로드
model_name = 'klue/bert-base'
tokenizer = AutoTokenizer.from_pretrained(model_name)

In [ ]:
# 전처리 함수 정의 : label을 float 타입으로 변환 포함
def preprocess_function(data):
  tokenized_outputs = tokenizer(data['text'], truncation=True, padding=True)
  labels_batch = np.array(data['multi_hot_label'], dtype=np.float32)
  tokenized_outputs['labels'] = labels_batch
  return tokenized_outputs

# 전처리 적용
train_ts = train_ts.map(preprocess_function, batched=True)
val_ts = val_ts.map(preprocess_function, batched=True)

## 3.Fine-Tuning1
* 일반 파인 튜닝으로

### (1) 사전학습 모델 준비
* problem_type="multi_label_classification"

In [ ]:
# label(클래스) 수 지정
n = 11

# 모델 준비 : 옵션 problem_type="multi_label_classification" 추가
model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=n,problem_type='multi_label_classification').to(device)

### (2) 학습 설정

* TrainArguments

In [ ]:
# TrainingArguments 설정
training_args = TrainingArguments(
    output_dir = './results',
    eval_strategy = "epoch",
    save_strategy = "epoch",
    learning_rate = 2e-8,
    per_device_train_batch_size = 32,
    per_device_eval_batch_size = 32,
    num_train_epochs = 5,
    weight_decay = 0.02,
    load_best_model_at_end = True,
    logging_dir ='./logs',
    logging_steps = 10,
    report_to="tensorboard"
)

* Trainer

In [ ]:
# 학습 시 추가 함수 준비
from torch.utils.data import DataLoader
from transformers import default_data_collator
from transformers import DataCollatorWithPadding

padding_collator = DataCollatorWithPadding(tokenizer=tokenizer)
# 평가함수(선택 사항)
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    probs = 1 / (1 + np.exp(-logits))  # sigmoid
    preds = (probs > 0.5).astype(int)
    return {
        "weighted_f1": f1_score(labels, preds, average="weighted"),
        "macro_f1": f1_score(labels, preds, average="macro")
    }

# 사용자 정의 data_collator(필수 사항)
def float_label_collator(features):
    # batch = default_data_collator(features)
    batch = padding_collator(features)
    batch["labels"] = batch["labels"].float()
    return batch

In [ ]:
# Trainer 설정 :
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset = train_ts,
    eval_dataset = val_ts,
    tokenizer = tokenizer,
    callbacks = [EarlyStoppingCallback(early_stopping_patience=3)],
    data_collator=float_label_collator,  # 필수사항
    compute_metrics=compute_metrics,     # 선택사항
)

### (3) 학습
* 7000건 학습데이터에 대해서, 학습 시간이 10~13분 정도 소요 됩니다.

In [ ]:
trainer.train()

### (4) 모델 검증평가

In [ ]:
eval_results = trainer.evaluate()

In [ ]:
print(f"Evaluation Results: {eval_results}")

* 예측값

In [ ]:
raw_pred = trainer.predict(val_ts)


In [ ]:
logits = raw_pred.predictions
probs = 1 / (1 + np.exp(-logits))
probs

In [ ]:
preds = np.where(probs > 0.5, 1, 0)

* 실제값

In [ ]:
true = raw_pred.label_ids

* 평가

In [ ]:
# average=None : 각 레이블 별로 분리해서 평가
precisions, recalls, f1s, supports = precision_recall_fscore_support(
    true, preds, average=None, zero_division=0)

# 출력
for i, label in enumerate(labels):
    print(f"{label:5} | Precision: {precisions[i]:.3f}, Recall: {recalls[i]:.3f}, F1: {f1s[i]:.3f}, Support: {supports[i]}")

In [ ]:
df = pd.DataFrame({
    "label": labels,
    "precision": precisions,
    "recall": recalls,
    "f1": f1s,
    "support": supports
})
df_melted = df.melt(id_vars='label', value_vars=['precision', 'recall', 'f1'],
                    var_name='metric', value_name='score')

In [ ]:
# barplot 그리기
plt.figure(figsize=(10, 5))
sns.barplot(data=df_melted, x='label', y='score', hue='metric')

# 3. 한글 폰트 설정 (필요 시)
plt.xticks(fontproperties=fontprop)
plt.title("속성 별 성능 비교 (정밀도, 재현율, F1)", fontproperties=fontprop)
plt.ylabel("Score", fontproperties=fontprop)
plt.xlabel("Label", fontproperties=fontprop)
plt.ylim(0, 1.2)
plt.legend(prop=fontprop)
plt.grid()
plt.show()

## 4.Fine-Tuning2
* LoRA 튜닝하기

### (1) 사전학습 모델 준비
* problem_type="multi_label_classification"

In [ ]:
base_model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=n,problem_type='multi_label_classification').to(device)

### (2) LoRA 구성

In [ ]:
# LoRA 구성 정의
lora_config = LoraConfig(
    task_type=TaskType.SEQ_CLS,
    r=8,
    lora_alpha=32,
    lora_dropout=0.1,
    target_modules=['query','value'],
    bias='none'
)

# LoRA 적용
model = get_peft_model(base_model, lora_config)



### (3) 학습 설정

* TrainArguments
    * 일반 파인 튜닝보다는 학습률을 조금 높게 설정

In [ ]:
training_args = TrainingArguments(
    output_dir = './results',
    eval_strategy = "epoch",
    save_strategy = "epoch",
    learning_rate = 2e-4,
    per_device_train_batch_size = 32,
    per_device_eval_batch_size = 32,
    num_train_epochs = 5,
    weight_decay = 0.02,
    load_best_model_at_end = True,
    logging_dir ='./logs',
    logging_steps = 10,
    report_to="tensorboard"
)

* Trainer

In [ ]:
# 사용자 정의 data_collator 생
from torch.utils.data import DataLoader
from transformers import default_data_collator

def float_label_collator(features):
    batch = padding_collator(features)
    batch["labels"] = batch["labels"].float()
    return batch

# Trainer 설정
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset = train_ts,
    eval_dataset = val_ts,
    tokenizer = tokenizer,
    data_collator = float_label_collator,
    callbacks = [EarlyStoppingCallback(early_stopping_patience=3)],
)

### (4) 학습

In [ ]:
trainer.train()

In [ ]:
eval_results = trainer.evaluate()
print(f"Evaluation results: {eval_results}")

### (5) 모델 검증평가

In [ ]:
# 예측
raw_pred = trainer.predict(val_ts)
logits = raw_pred.predictions
probs = 1 / (1 + np.exp(-logits))
preds = (probs > 0.5).astype(int)
true = raw_pred.label_ids

In [ ]:

precisions, recalls, f1s, supports = precision_recall_fscore_support(
    true, preds, average = None, zero_division=0
)


for i, label in enumerate(labels):
    print(f"{label:5} | Precision: {precisions[i]:.3f}, Recall: {recalls[i]:.3f}, F1: {f1s[i]:.3f}, Support: {supports[i]}")

In [ ]:
df = pd.DataFrame({
    "label": labels,
    "precision": precisions,
    "recall": recalls,
    "f1": f1s,
    "support": supports
})
df_melted = df.melt(id_vars='label', value_vars=['precision', 'recall', 'f1'],
                    var_name='metric', value_name='score')


plt.figure(figsize=(10, 5))
sns.barplot(data=df_melted, x='label', y='score', hue='metric')


plt.xticks(fontproperties=fontprop)
plt.title("속성 별 성능 비교 (정밀도, 재현율, F1)", fontproperties=fontprop)
plt.ylabel("Score", fontproperties=fontprop)
plt.xlabel("Label", fontproperties=fontprop)
plt.ylim(0, 1.2)
plt.legend(prop=fontprop)
plt.grid()
plt.show()